***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.4 连续傅里叶变换：从信号表示到可见度语言](2_4_the_fourier_transform.ipynb)
    * 下一节：[2.6 互相关、自相关与相似性度量](2_6_cross_correlation_and_auto_correlation.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 2.5 卷积：响应函数如何重塑信号<a id='math:sec:convolution'></a>

傅里叶变换告诉我们如何把信号分解成频率分量；卷积告诉我们一个系统如何把输入信号重塑成输出信号。在射电观测中，望远镜并不直接给出真实天空亮度，而是给出被主波束、采样函数、积分时间、频率通道和成像点扩散函数共同改写后的结果。卷积是描述这种“真实分布经过响应函数后被展宽、平滑或带上旁瓣”的基本运算。

本节先从线性系统的单位脉冲响应出发，推导卷积积分；随后说明它的几何意义、代数性质和傅里叶域解释；最后用一维图像说明卷积为什么会造成平滑、信息损失和点源展宽。这些直觉会直接进入后面关于脏图像与反卷积的讨论。


### 2.5.1 从单位脉冲响应到卷积积分<a id='math:sec:definition_of_convolution'></a>

设一个系统对位于原点的单位脉冲 $\delta(x)$ 的响应为 $g(x)$。若输入不是原点脉冲，而是位于 $t$ 的脉冲 $\delta(x-t)$，并且系统具有平移不变性，那么输出就是同一个响应函数平移到 $t$：

$$
\delta(x-t)\longrightarrow g(x-t).
$$

任意函数 $f(t)$ 可以看作许多位置上的小脉冲叠加而成。位置 $t$ 附近的输入强度为 $f(t)dt$，它对输出位置 $x$ 的贡献为 $f(t)g(x-t)dt$。把所有位置的贡献加起来，就得到卷积：

<a id='math:eq:convolution_definition'></a><!--\label{math:eq:convolution_definition}-->
$$
(f*g)(x)=\int_{-\infty}^{+\infty}f(t)g(x-t)\,dt.
$$

通过变量替换 $u=x-t$，也可写成

$$
(f*g)(x)=\int_{-\infty}^{+\infty}f(x-u)g(u)\,du.
$$

这两个写法等价。第一种更接近“每个输入点激发一个响应”的物理图像；第二种更接近“响应函数滑过输入函数”的几何图像。


卷积的多维形式完全类似。对二维图像，

<a id='math:eq:multidimensional_convolution'></a><!--\label{math:eq:multidimensional_convolution}-->
$$
(f*g)(\mathbf{x})
=\int_{\mathbb{R}^n}f(\mathbf{t})g(\mathbf{x}-\mathbf{t})\,d^n\mathbf{t}.
$$

在干涉成像中，$\mathbf{x}$ 可以是天球切平面坐标 $(l,m)$，$g$ 可以是合成波束或点扩散函数。若暂时忽略主波束、噪声和非共面基线等效应，脏图像常被写成

$$
I_{\rm D}(l,m)\approx I(l,m)*B_{\rm D}(l,m),
$$

其中 $B_{\rm D}$ 是脏波束。这个式子表达的不是一个细节，而是成像问题的核心：观测图像中的每个点源都会被替换成一个波束形状。


### 2.5.2 滑动叠加的几何图像<a id='math:sec:sliding_overlap'></a>

一维卷积可以按“翻转、平移、相乘、积分”的方式理解。若使用

$$
(f*g)(x)=\int f(t)g(x-t)dt,
$$

则在每个输出位置 $x$，函数 $g$ 被反向并平移，随后与 $f$ 相乘，最后对重叠面积积分。当 $g$ 是偶函数时，反向操作看不出来；当 $g$ 非对称时，卷积与下一节的互相关会给出不同结果。

![卷积的滑动重叠图像](figures/convolution_sliding_overlap.png)

**图 2.5.1** 卷积的滑动重叠解释。对每一个输出位置，响应函数与输入函数的重叠面积给出卷积结果在该位置的取值。


这个图像解释了为什么卷积会平滑信号。如果响应函数 $g$ 具有有限宽度，那么输出 $(f*g)(x)$ 不只依赖 $f(x)$，还依赖 $x$ 附近一段范围内的输入。响应函数越宽，参与平均的范围越大，细节越容易被抹平。望远镜角分辨率有限、频率通道宽度有限、时间积分长度有限，都可以用同一类语言理解。


### 2.5.3 卷积的基本性质<a id='math:sec:properties_of_convolution'></a>

卷积是线性运算，并且满足交换律、结合律和分配律：

<a id='math:eq:convolution_properties'></a><!--\label{math:eq:convolution_properties}-->
$$
\begin{aligned}
f*g&=g*f,\\
(f*g)*h&=f*(g*h),\\
f*(g+h)&=f*g+f*h,\\
(a f)*g&=a(f*g).
\end{aligned}
$$

这些性质使得复杂系统可以被拆成若干响应函数的组合。例如，若一个观测过程先受到主波束平滑，再受到有限积分时间平滑，在理想平移不变近似下，可以把总响应写成两个响应函数的卷积。结合律保证“先合并响应函数再作用于信号”和“逐步作用于信号”是等价的。

单位元是 delta 函数：

<a id='math:eq:delta_identity_convolution'></a><!--\label{math:eq:delta_identity_convolution}-->
$$
f*\delta=f.
$$

移位 delta 会把函数平移：

<a id='math:eq:shifted_delta_convolution'></a><!--\label{math:eq:shifted_delta_convolution}-->
$$
f*\delta(x-x_0)=f(x-x_0).
$$

这一点是理解点源成像的关键。若天空由多个点源组成，与点扩散函数卷积后，观测图像就是多个平移后的点扩散函数叠加。


交换律可以用一个简单变量替换验证：

$$
\begin{aligned}
(f*g)(x)&=\int_{-\infty}^{+\infty}f(t)g(x-t)dt.\\
\end{aligned}
$$

令 $u=x-t$，则 $dt=-du$，积分上下限交换后得到

$$
(f*g)(x)=\int_{-\infty}^{+\infty}g(u)f(x-u)du=(g*f)(x).
$$

结合律和分配律可通过交换积分次序和展开括号得到。真正重要的是物理含义：卷积能够自然描述许多局域响应的叠加，并且与线性组合相容。


### 2.5.4 卷积定理的物理含义<a id='math:sec:convolution_theorem_link'></a>

卷积定理指出

<a id='math:eq:convolution_theorem_preview'></a><!--\label{math:eq:convolution_theorem_preview}-->
$$
\mathcal{F}\{f*g\}=\mathcal{F}\{f\}\,\mathcal{F}\{g\}.
$$

也就是说，原始域中的卷积，在傅里叶域中变成逐点乘法。反过来，原始域中的乘法，在傅里叶域中变成卷积：

$$
\mathcal{F}\{fg\}=F*G.
$$

这条定理是干涉成像的核心工具之一。图像域中与波束卷积，等价于 `uv` 域中乘以采样或权重函数；图像域中乘以主波束，则等价于 `uv` 域中与孔径照明函数卷积。第 2.7 节会给出更系统的傅里叶定理推导，本节先保留这个物理图像。


### 2.5.5 平滑、噪声与分辨率损失<a id='math:sec:convolution_smoothing'></a>

卷积常被用于平滑含噪数据。若噪声在相邻采样点之间近似不相关，而响应函数覆盖多个采样点，那么卷积会把噪声部分平均掉；但同一个操作也会抹平真实信号中的窄结构。这是一种典型的方差与分辨率折中。

![卷积平滑的折中](figures/convolution_smoothing_tradeoff.png)

**图 2.5.2** 用不同宽度的核对含噪信号做卷积。较宽的核更能压低随机起伏，但也更容易削弱窄结构和快速变化。


在射电图像处理中，这种折中经常表现为“把图像恢复到某个共同分辨率”。例如比较不同频率或不同阵列配置的图像时，常需要把高分辨率图像卷积到低分辨率波束，以避免把分辨率差异误认为真实谱指数或形态差异。卷积不是无害的显示操作，它会改变图像中可独立辨认的结构尺度。


### 2.5.6 点源、点扩散函数与脏图像<a id='math:sec:point_sources_psf'></a>

若理想天空是一组点源，

$$
I(x)=\sum_k S_k\delta(x-x_k),
$$

与点扩散函数 $B(x)$ 卷积后得到

<a id='math:eq:point_sources_convolved_psf'></a><!--\label{math:eq:point_sources_convolved_psf}-->
$$
(I*B)(x)=\sum_k S_k B(x-x_k).
$$

每个点源都变成一个以自身位置为中心、幅度按通量缩放的 PSF 副本。若 PSF 有旁瓣，所有点源的旁瓣会相互叠加，造成脏图像中的复杂结构。CLEAN 等反卷积方法正是利用这个模型，反复寻找点源分量并减去相应的脏波束响应。

![点源与点扩散函数的卷积](figures/convolution_point_sources_psf.png)

**图 2.5.3** 点源与非对称点扩散函数卷积后的观测图像。每个点源都被替换为一个平移后的 PSF，旁瓣会随源强一起进入图像。


### 2.5.7 与互相关的区别<a id='math:sec:convolution_vs_correlation'></a>

卷积和互相关看起来相似，但二者的几何操作不同。卷积包含响应函数的反向和平移，适合描述“输入经过系统响应后产生输出”；互相关通常不做这一步反向，而是衡量两个函数在不同相对平移下的相似程度。若响应函数是偶函数，二者可能给出相同形状；若函数非对称，差异就很明显。

下一节会专门讨论互相关与自相关。射电干涉测量中的相关器首先计算的是电场之间的相关；成像中的脏图像与脏波束关系则通常以卷积形式表达。把这两个概念分清，是理解相关测量与图像响应之间关系的必要步骤。


***

* 下一节：[2.6 互相关、自相关与相似性度量](2_6_cross_correlation_and_auto_correlation.ipynb)
